# Bike-Demand - Colab runner (train -> per-city eval)

HUJI IML 67577, Challenge 1. Runs the **official-data** workflow end-to-end on a fresh
Colab VM: clone -> install -> load `train_set.csv` from Drive -> local time-split -> train ->
per-city MAE -> save outputs back to Drive.

**Notes**
- **CPU only** (LightGBM/XGBoost tree models) - no GPU needed (`Runtime > Change runtime type > CPU`).
- The Colab VM is **ephemeral**: outputs are saved to **Google Drive** so they survive a disconnect.
- **Edit one thing:** `DRIVE_CSV` (Section 4). Optionally `REPO_URL`/`BRANCH` (Section 2) and `SUBMISSION`/`FAST` (Section 6).
- Official data only - **cities 1-3, no Chicago / external enrichment** here.
- Use a **High-RAM** runtime only if you later train on much bigger data.

In [ ]:
# === Section 2: clone the repo ===========================================
REPO_URL = "https://github.com/GuyShabat7/HUJI-IML-Hackathon.git"  # <- or your fork
BRANCH   = "main"

import os, subprocess
REPO_DIR = "/content/HUJI-IML-Hackathon"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-1"])

In [ ]:
# === Section 3: install dependencies =====================================
!pip -q install lightgbm xgboost joblib pandas numpy scikit-learn
import lightgbm, xgboost, joblib, pandas, numpy, sklearn
print("lightgbm", lightgbm.__version__, "| xgboost", xgboost.__version__,
      "| pandas", pandas.__version__, "| numpy", numpy.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# === Section 4: load train_set.csv from Google Drive =====================
# Leave DRIVE_CSV = "" to AUTO-FIND train_set.csv anywhere in your Drive.
# (Or set an exact path, e.g. "/content/drive/MyDrive/iml_hackathon/train_set.csv".)
DRIVE_CSV = ""

from google.colab import drive
drive.mount("/content/drive")

import os, glob, shutil
os.makedirs("dataset", exist_ok=True)
DST = "dataset/train_set.csv"

src = None
if DRIVE_CSV and os.path.exists(DRIVE_CSV):
    src = DRIVE_CSV
else:
    print("searching your Drive for train_set.csv ...")
    hits = glob.glob("/content/drive/MyDrive/**/train_set.csv", recursive=True)
    if hits:
        src = hits[0]
        print("auto-found:", src, "" if len(hits) == 1 else f"(+{len(hits)-1} more; using the first)")

if src:
    shutil.copy(src, DST)
else:
    print("train_set.csv not found in Drive - upload it from your computer now:")
    from google.colab import files
    up = files.upload()                       # pick train_set.csv
    open(DST, "wb").write(next(iter(up.values())))

import pandas as pd
n_rows = sum(1 for _ in open(DST)) - 1
print(f"rows: {n_rows:,} -> {DST}")
print("cities:", sorted(pd.read_csv(DST, usecols=["city"]).city.unique()))

In [ ]:
# === Section 5: local time-split + build evaluator-format files ==========
# Time-based holdout per city (latest ~20%), then the station-hour eval format.
!python make_local_split.py
!python build_station_hour_eval_data.py --input_csv dataset/local_validation_set.csv --public_targets_csv dataset/public_validation_targets.csv --private_labels_csv dataset/private_labels.csv

import os
for f in ["dataset/local_train_set.csv", "dataset/local_validation_set.csv",
          "dataset/public_validation_targets.csv", "dataset/private_labels.csv"]:
    print("ok " if os.path.exists(f) else "MISSING ", f)

In [ ]:
# === Section 6: train ====================================================
SUBMISSION = "submissions/challenge_1_IDs"      # or "submissions/challenge_1_ensamble" (XGBoost ensemble)
FAST       = False                               # True = quick iteration (small trees); honored by the ensemble

import os, time, subprocess
cmd = ["python", "train.py"]
if FAST:
    cmd.append("--fast")   # the ensemble honors --fast; the LightGBM baseline simply ignores it
print("running:", " ".join(cmd), "in", SUBMISSION)
t0 = time.time()
subprocess.run(cmd, cwd=SUBMISSION, check=True)
print()
print(f"trained {SUBMISSION} in {time.time()-t0:.1f}s")
assert os.path.exists(os.path.join(SUBMISSION, "weights.joblib")), "train.py did not produce weights.joblib"

In [ ]:
# === Section 7: per-city MAE =============================================
# Official evaluator on the local validation targets built in Section 5.
# (It scans submissions/; only folders with a weights.joblib score - others are skipped.)
!python evaluate.py --eval_dir dataset --submissions_dir submissions --test_targets_csv public_validation_targets.csv --test_labels_csv private_labels.csv --output_csv mae_by_city.csv

import pandas as pd
print()
print(pd.read_csv("mae_by_city.csv").to_string(index=False))

# NOTE: train.py trains on the FULL train_set.csv, so this validation slice overlaps training
# (optimistic). For an honest held-out per-city + blended number (model trained on c1+c2,
# city 3 unseen), use the repo harness instead:
# !python tools/eval_local.py {SUBMISSION}

In [ ]:
# === Section 8: save outputs to Drive ====================================
import os, shutil
OUT_DIR = "/content/drive/MyDrive/iml_hackathon/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
saved = []
w = os.path.join(SUBMISSION, "weights.joblib")
if os.path.exists(w):
    dst = os.path.join(OUT_DIR, f"weights_{os.path.basename(SUBMISSION)}.joblib")
    shutil.copy(w, dst); saved.append(dst)
if os.path.exists("mae_by_city.csv"):
    dst = os.path.join(OUT_DIR, "mae_by_city.csv")
    shutil.copy("mae_by_city.csv", dst); saved.append(dst)
print("saved to Drive:")
for s in saved:
    print("  ", s)